# PostgreSQL 2~6교시 핵심 개념 데모

> 2교시(왜 관계형 DB인가)부터 6교시(EXPLAIN ANALYZE 실측)까지 — 교안의 핵심 데모를 실제 PostgreSQL 위에서 그대로 재현합니다. **부록 A와 같은 방식**입니다: 빈칸을 채우는 실습이 아니라, 셀을 실행하면 결과가 바로 보이는 시연/테스트용 노트북입니다.

## 차시 학습 목표
- (2교시) CSV/파일 기반 관리의 "검색 속도" 벽을, 키 형식이 다르면 **결과가 조용히 틀려지는** 사례로 직접 재현합니다.
- (3교시) `CHECK` 제약이 실제로 잘못된 값을 막는 것을 실행 오류로 확인합니다.
- (4교시) `INNER JOIN`·`LEFT JOIN`의 차이와 B-tree 인덱스 생성을 직접 실행합니다.
- (5교시) `pg_stats`로 옵티마이저가 참고하는 통계(선택도 판단 근거)를 조회합니다.
- (6교시) `EXPLAIN ANALYZE`로 "인덱스가 안 타는 3가지 조건"을 실측하고, 통계가 오래됐을 때의 예측 오차를 직접 만들어봅니다.

## 다루는 내용
- CSV의 세 가지 벽 — 키 형식 불일치로 결과가 조용히 틀리는 사례 재현 (2교시)
- 관계형 모델 — `CHECK` 제약 위반 실측 (3교시)
- JOIN — `INNER`/`LEFT JOIN`, B-tree 인덱스 생성 (4교시)
- 옵티마이저 — `pg_stats`(n_distinct·correlation) (5교시)
- `EXPLAIN ANALYZE` — 가설 1·2·3 실측, 함수형 인덱스·범위 재작성 해결법, 통계 드리프트 전후 비교 (6교시)

## 실행 안내
- 🐳 **선행 조건**: 1교시에서 `db-pg` 컨테이너를 띄운 상태를 그대로 이어갑니다. (아래 '연결 준비' 셀이 없으면 자동으로 만듭니다 — 재실행 안전)
- 이 노트북은 **7~8교시보다 먼저, 또는 독립적으로** 실행할 수 있도록 `course_db`를 직접 만듭니다(없을 때만). 여기서 만드는 `hourly_error_stats`·`spike_windows`는 7교시 실습(`lab/load/skeleton|solution/schema.sql`)을 실행하는 순간 `DROP TABLE IF EXISTS`로 정리되고 실제 데이터로 다시 채워지므로, 이 노트북을 7교시보다 먼저 돌리든 나중에 돌리든 순서를 걱정하지 않아도 됩니다.
- 6교시 절에서 100만 행짜리 `log_events` 테이블을 직접 만듭니다 — `lab/load/data/init_log_events.sql`을 먼저 실행해 뒀다면 그 테이블을 그대로 재사용합니다(재실행 안전, 없으면 이 노트북이 새로 만듭니다).
- ⏱️ **`EXPLAIN ANALYZE`의 정확한 ms 값과 실행 계획(Index Scan/Bitmap Heap Scan 등)은 컴퓨터·PostgreSQL 버전마다 다르게 나옵니다.** 이 노트북의 결과값은 제작 환경(PostgreSQL 16)에서 실측한 값이며, 목적은 정확히 같은 숫자를 재현하는 것이 아니라 **대소 관계와 선택도 판단 원리가 교안과 같은 방향인지**를 확인하는 것입니다.
- 🔐 비밀번호는 `.env`에서 `load_dotenv()`로 입력받습니다(하드코딩 금지).


In [1]:
# 🪟 Windows 콘솔의 문자 인코딩을 UTF-8(65001)로 바꿉니다.
# 📖 문법 참고: chcp는 원래 cmd.exe 명령이라 파이썬 코드가 아니지만, os.system(문자열)을 쓰면
#    그 문자열을 셸 명령으로 그대로 실행할 수 있습니다. 이걸 안 하면 아래 !docker ... 명령이
#    출력하는 한글이 깨져 보일 수 있습니다(화면에 뜨는 "Active code page: 65001" 같은 안내 문구는
#    무시해도 됩니다 — 실행 결과이지 에러가 아닙니다).
import os
os.system('chcp 65001')

0

## 연결 준비 — course_db에 psycopg로 연결

2~6교시 데모에서 쓸 `course_db`에 연결합니다. (7~8교시 실습에서도 같은 DB를 씁니다 — 어느 쪽을 먼저 실행해도 안전하도록 `CREATE DATABASE`는 없을 때만 실행됩니다.)


In [2]:
# 💡 사전 준비(재실행 안전): db-pg가 실행 중인지 확인하고, course_db가 없으면 만듭니다.
# 아래는 Windows(cmd.exe) 기준입니다 — Linux/macOS는 바로 아래 주석 처리된 줄을 대신 쓰세요.
# 📖 문법 참고: A && B는 "A가 성공해야 B도 실행", A || B는 "A가 실패해야 B를 실행"이라는 뜻입니다.
#    psql -tc "SQL"의 -t는 헤더·구분선 없이 결과값만, -c는 그 SQL 하나만 실행하고 끝내라는 옵션입니다.
#    findstr /C:"1" >NUL은 출력에서 문자 "1"을 찾되(>NUL로 화면 출력은 숨김) "찾았는지 여부"만
#    성공/실패로 남깁니다. 즉 이 두 줄은 "course_db가 이미 있으면(SELECT 결과가 1) 그냥 넘어가고,
#    없으면 CREATE DATABASE로 새로 만든다"는 뜻입니다 — 여러 번 실행해도 안전한 이유입니다.
!docker start db-pg >NUL 2>&1 && echo db-pg 실행 중
!docker exec db-pg psql -U postgres -tc "SELECT 1 FROM pg_database WHERE datname='course_db'" | findstr /C:"1" >NUL || docker exec db-pg psql -U postgres -c "CREATE DATABASE course_db"

# ---- Linux/macOS ----
# !docker start db-pg >/dev/null 2>&1 && echo "db-pg 실행 중"
# !docker exec db-pg psql -U postgres -tc "SELECT 1 FROM pg_database WHERE datname='course_db'" | grep -q 1 || docker exec db-pg psql -U postgres -c "CREATE DATABASE course_db"


db-pg 실행 중
CREATE DATABASE


In [3]:
# ✅ 포인트: 1교시 'Python에서 연결'과 같은 방식으로 course_db에 연결합니다. 비밀번호는 .env에서 load_dotenv()로(하드코딩 금지).
# 📖 문법 참고: psycopg.connect(...)로 만든 conn은 서버로 가는 "전화선"입니다. conn.autocommit = False로
#    두면 INSERT/UPDATE/DELETE를 실행해도 conn.commit()을 호출하기 전까지는 실제로 저장되지 않습니다
#    (잘못 넣었을 때 conn.rollback()으로 취소할 수 있는 안전장치). conn.cursor()가 만드는 cur는
#    SQL을 실제로 실행하는 도구입니다 — conn(연결)과 cur(커서)는 역할이 다릅니다.
import psycopg, os
from dotenv import load_dotenv
load_dotenv()
pw = os.environ["PGPASSWORD"]
conn = psycopg.connect(host="localhost", port=5432, dbname="course_db", user="postgres", password=pw)
conn.autocommit = False
cur = conn.cursor()

def run(sql):
    """SQL을 실행하고 결과 행을 한 줄씩 출력한 뒤, 행 리스트를 돌려줍니다."""
    # 📖 cur.execute(sql)은 SQL을 서버로 보내 실행만 합니다(결과를 돌려주지 않음).
    #    결과를 받아오려면 그다음에 cur.fetchall()을 따로 호출해야 합니다 — SELECT처럼 결과가 있는
    #    쿼리에서만 의미가 있고, INSERT/CREATE처럼 결과가 없는 SQL 뒤에 fetchall()을 부르면 에러가 납니다.
    cur.execute(sql)
    rows = cur.fetchall()
    for r in rows:
        print(r)
    print(f"  → {len(rows)}행")
    return rows

print("course_db 연결 완료 →", conn.info.dbname)


course_db 연결 완료 → course_db


---
## 2교시 · CSV 관리의 세 가지 벽 — "벽 3(검색 속도)"를 직접 재현

파일(CSV/JSON) 기반 관리는 **동시 수정·무결성 보장·검색 속도** 세 가지 벽에 부딪힙니다(교안 모듈 2-1). 여기서는 세 번째 벽을 팀별 결과 파일 3개로 직접 재현합니다.

> 실제 규모(1만 파일·24만 행)에서 측정한 교안의 비교표는 다음과 같습니다 — 이 노트북에서 그 규모까지 재현하지는 않고, **왜** 이런 차이가 나는지의 구조만 작은 규모로 확인합니다.

| 방식 | 실행 시간 | 비고 |
| --- | --- | --- |
| 파일 1만 개 순회 (Python) | 149.5ms | 파일마다 디스크 I/O |
| DB, 인덱스 없음 (Seq Scan) | 17.1ms | 파일 open/close 비용 없음 |
| DB, `hour`에 인덱스 (Bitmap Heap Scan) | 3.8ms | 인덱스로 후보 행 자체를 줄임 |


In [4]:
# 💡 팀별 결과 파일 3개를 만듭니다. team_C만 시간 키를 "03"이 아니라 "3"으로 저장했습니다(0으로 안 채움) — 일부러 형식을 다르게 뒀습니다.
# 📖 문법 참고: Path("results_demo")는 폴더 경로를 표현하는 객체이고, mkdir(exist_ok=True)는
#    "폴더가 없으면 만들고, 이미 있어도 에러 내지 않기"라는 뜻입니다. json.dumps(딕셔너리)는
#    파이썬 딕셔너리를 JSON 문자열로 바꿔주고, write_text(...)는 그 문자열을 파일에 그대로 씁니다.
#    {"team": "A", "hourly_errors": {"00": ..., "03": ...}}처럼 딕셔너리 안에 딕셔너리를 넣어
#    중첩 구조로 데이터를 표현했습니다.
import json
from pathlib import Path

demo_dir = Path("results_demo")
demo_dir.mkdir(exist_ok=True)
(demo_dir / "team_A.json").write_text(json.dumps({"team": "A", "hourly_errors": {"00": 7148, "03": 61442, "14": 12088}}), encoding="utf-8")
(demo_dir / "team_B.json").write_text(json.dumps({"team": "B", "hourly_errors": {"00": 3021, "03": 4110,  "14": 2870}}), encoding="utf-8")
(demo_dir / "team_C.json").write_text(json.dumps({"team": "C", "hourly_errors": {"0":  5200, "3":  58900, "14": 9100}}), encoding="utf-8")

# ❌ 교안 그대로 — 파일을 하나씩 열어 "03" 키로 검색 (data["hourly_errors"].get("03", 0) > 50_000)
# 📖 문법 참고: demo_dir.glob("*.json")은 그 폴더 안에서 이름이 .json으로 끝나는 파일을 전부 찾아주고,
#    sorted(...)로 이름 순서를 고정합니다. json.loads(문자열)은 JSON 문자열을 다시 파이썬 딕셔너리로
#    되돌립니다(dumps의 반대). dict.get("03", 0)은 "03" 키가 있으면 그 값을, 없으면 기본값 0을
#    돌려줍니다 — 에러 없이 조용히 넘어간다는 뜻이고, 이게 바로 아래에서 team_C를 놓치는 원인입니다.
results = []
for p in sorted(demo_dir.glob("*.json")):
    data = json.loads(p.read_text(encoding="utf-8"))
    if data["hourly_errors"].get("03", 0) > 50_000:
        results.append(p.name)

print("검색 결과 (5만 건 초과 팀):", results)


검색 결과 (5만 건 초과 팀): ['team_A.json']


### ⚠️ 흔한 실수가 실제로 일어나는 순간

방금 결과에 `team_A.json`만 나왔습니다. 그런데 `team_C`도 **58,900건**으로 5만 건을 넘습니다 — 왜 빠졌을까요? 아래 셀에서 세 파일의 실제 키를 그대로 출력해 봅니다.


In [5]:
# ⚠️ team_C는 시간 키가 "03"이 아니라 "3"이라 .get("03", 0)이 기본값 0을 돌려줍니다 — 에러 없이 조용히 놓칩니다.
for p in sorted(demo_dir.glob("*.json")):
    data = json.loads(p.read_text(encoding="utf-8"))
    print(f"{p.name}: hourly_errors = {data['hourly_errors']}")


team_A.json: hourly_errors = {'00': 7148, '03': 61442, '14': 12088}
team_B.json: hourly_errors = {'00': 3021, '03': 4110, '14': 2870}
team_C.json: hourly_errors = {'0': 5200, '3': 58900, '14': 9100}


### 같은 데이터를 DB에 넣고 같은 조건으로 찾으면

이번에는 세 팀의 같은 데이터를 표 하나(`team_hourly_results`)에 넣고, SQL로 같은 조건을 찾습니다. `hour` 컬럼이 **SMALLINT 타입**이므로 "03"과 "3"이라는 표기 차이 자체가 애초에 존재하지 않습니다.

> `team_hourly_results`는 이 절의 "벽 3"만 보여주기 위한 임시 데모 테이블입니다 — 7교시에서 실제로 만드는 `hourly_error_stats`(파일명 없이 날짜·시간대로 이미 집계된 24행짜리 표)와는 스키마가 다릅니다. 이름이 비슷해 보여도 다른 표입니다.


In [6]:
# ✅ 포인트: hour가 정수 컬럼이므로 "03"/"3" 같은 표기 차이가 아예 생기지 않습니다 — 무결성(벽 2)이 검색(벽 3)의 정확성까지 함께 지켜줍니다.
# 📖 문법 참고: SMALLINT·INTEGER·TEXT는 컬럼의 자료형(타입)입니다 — SMALLINT는 작은 정수,
#    TEXT는 길이 제한 없는 문자열입니다. executemany(SQL, [행1, 행2, ...])는 같은 SQL을 여러 행에
#    대해 반복 실행합니다 — SQL 안의 %s는 "이 자리에 파이썬 값을 넣어라"는 자리표시자(placeholder)이고,
#    각 튜플의 값이 순서대로 %s에 채워집니다. f-string이나 문자열 이어붙이기로 값을 직접 끼워넣지 않는
#    이유는 SQL 인젝션을 막기 위해서입니다 — psycopg가 안전하게 값을 이스케이프해서 넣어줍니다.
cur.execute("DROP TABLE IF EXISTS team_hourly_results")
cur.execute("""
    CREATE TABLE team_hourly_results (
        file_name   TEXT,
        hour        SMALLINT,
        error_count INTEGER
    )
""")
cur.executemany(
    "INSERT INTO team_hourly_results (file_name, hour, error_count) VALUES (%s, %s, %s)",
    [("team_A.json", 3, 61442), ("team_B.json", 3, 4110), ("team_C.json", 3, 58900)],
)
conn.commit()

run("SELECT file_name FROM team_hourly_results WHERE hour = 3 AND error_count > 50000 ORDER BY file_name")


('team_A.json',)
('team_C.json',)
  → 2행


[('team_A.json',), ('team_C.json',)]

### 💡 핵심 요약

- 파일(JSON) 검색은 **키 형식이 흐트러지면 조용히 틀린 결과**를 냅니다 — `team_C`의 진짜 급증(58,900건)을 그냥 놓쳤습니다. 에러가 안 나서 알아채기도 더 어렵습니다.
- DB는 `hour`를 처음부터 정수로 강제하므로 이런 표기 불일치 자체가 발생하지 않습니다 — **무결성이 검색 정확성의 전제 조건**이라는 것을 보여줍니다.
- 속도(149.5ms → 3.8ms)뿐 아니라 **정확성**도 DB 쪽이 구조적으로 유리합니다.


In [7]:
# 🧹 정리 — 이 절만을 위한 임시 테이블이므로 정리합니다 (7교시 실습과 무관)
cur.execute("DROP TABLE IF EXISTS team_hourly_results")
conn.commit()
print("team_hourly_results 정리 완료")


team_hourly_results 정리 완료


---
## 3교시 · 관계형 모델 — 테이블·키·제약조건을 DDL로 확인

`hourly_error_stats`를 실제로 만들고, `CHECK` 제약이 잘못된 값을 정말로 막는지 **일부러 실패시켜** 확인합니다. 이 표는 이후 4·5교시 데모에서도 그대로 이어 씁니다(7교시에서 실제 데이터로 다시 채워지기 전까지).


In [8]:
# ✅ 포인트: SERIAL=자동증가 PK, CHECK로 0~23만 허용, UNIQUE로 날짜+시간 중복을 막습니다.
# 📖 문법 참고: SERIAL PRIMARY KEY는 "값을 안 넣으면 1,2,3...으로 자동으로 매기는 기본키"입니다.
#    CHECK (hour BETWEEN 0 AND 23)는 그 컬럼에 들어올 수 있는 값의 범위를 DB가 직접 강제하는
#    제약조건입니다 — 0~23 밖의 값을 넣으려 하면 INSERT 자체가 실패합니다(바로 다음 셀에서 확인).
#    UNIQUE (log_date, hour)는 "이 두 컬럼의 조합이 같은 행이 두 개 있으면 안 된다"는 제약입니다
#    (하나씩 따로는 중복 가능, 둘을 합친 조합만 유일해야 함).
cur.execute("DROP TABLE IF EXISTS spike_windows")
cur.execute("DROP TABLE IF EXISTS hourly_error_stats")
cur.execute("""
    CREATE TABLE hourly_error_stats (
        id          SERIAL PRIMARY KEY,
        log_date    DATE NOT NULL,
        hour        SMALLINT NOT NULL
                        CHECK (hour BETWEEN 0 AND 23),
        error_count INTEGER NOT NULL DEFAULT 0,
        total_count INTEGER NOT NULL DEFAULT 0,
        UNIQUE (log_date, hour)
    )
""")
conn.commit()
print("hourly_error_stats 생성 완료")


hourly_error_stats 생성 완료


### ⚠️ 일부러 실패시키는 셀 — hour=25는 CHECK 위반이어야 합니다

아래 INSERT는 **의도적으로 실패**합니다. psycopg는 오류가 나면 트랜잭션을 "실패 상태"로 두므로, 다음 명령을 쓰기 전에 `conn.rollback()`으로 되돌려야 합니다(교안 7교시에서 배우는 트랜잭션 원자성의 예고편입니다).


In [9]:
# ⚠️ 흔한 실수: 이 INSERT는 CHECK 제약 때문에 실패해야 정상입니다. 실패 후 rollback() 없이 다음 SQL을 실행하면
#    "current transaction is aborted" 오류가 연쇄적으로 납니다 — 반드시 rollback()으로 정리합니다.
# 📖 문법 참고: try/except는 "try 블록을 실행하다가 에러가 나면 프로그램을 멈추는 대신 except로
#    넘어가서 처리하라"는 뜻입니다. psycopg.errors.CheckViolation은 "CHECK 제약을 위반했다"는
#    특정 종류의 에러(예외)만 잡습니다(다른 종류의 에러라면 여기서 안 잡히고 그대로 발생합니다).
#    PostgreSQL은 한 트랜잭션 안에서 에러가 나면 그 트랜잭션 전체를 "중단(aborted)" 상태로
#    표시합니다 — conn.rollback()을 호출해 트랜잭션을 되돌려야 다음 SQL을 다시 실행할 수 있습니다.
try:
    cur.execute(
        "INSERT INTO hourly_error_stats (log_date, hour, error_count, total_count) VALUES (%s, %s, %s, %s)",
        ("2016-11-09", 25, 100, 1000),
    )
    conn.commit()
    print("INSERT 성공 — 이러면 안 됩니다!") #hour를 0부터 23까지로 제한했기 때문
except psycopg.errors.CheckViolation as e:
    conn.rollback()
    print("✅ 예상대로 CHECK 제약 위반으로 거부됨")
    print("  ", e.diag.message_primary)
    print("  DETAIL:", e.diag.message_detail)


✅ 예상대로 CHECK 제약 위반으로 거부됨
   new row for relation "hourly_error_stats" violates check constraint "hourly_error_stats_hour_check"
  DETAIL: Failing row contains (1, 2016-11-09, 25, 100, 1000).


### 💡 핵심 요약

- `CHECK (hour BETWEEN 0 AND 23)`가 실제로 잘못된 값을 거부하는 것을 확인했습니다.
- psycopg는 오류 후 트랜잭션이 "실패 상태"로 남습니다 — `conn.rollback()`으로 되돌려야 다음 명령이 정상 실행됩니다. (7교시에서 이 원리를 ACID의 **원자성**으로 정식으로 배웁니다.)
- 지금 `hourly_error_stats`는 **0행**입니다(실패한 INSERT는 반영되지 않았으므로) — 다음 절(4교시)에서 JOIN 데모용으로 몇 행을 넣습니다.


---
## 4교시 · 표를 잇고, 빠르게 찾는다

### 📦 모듈 4-1 · JOIN — INNER vs LEFT

교안의 4행짜리 예시 그대로 `hourly_error_stats`에 4행을 넣고, `spike_windows`(급증 구간 — 이 절만을 위한 간단한 버전)를 만들어 JOIN 차이를 직접 봅니다.

> 여기서 만드는 `spike_windows(id, start_hour, spike_ratio)`는 **3컬럼짜리 교육용 버전**입니다. 7교시에서 실제로 쓰는 `spike_windows`는 `log_date`·`end_hour`·`peak_count`가 더 있는 정식 스키마이며, 7교시 스크립트가 실행되는 순간 `DROP TABLE IF EXISTS`로 이 임시 버전을 정리하고 다시 만듭니다.


In [10]:
# ✅ 포인트: 교안 예시와 동일한 4행 + spike_windows 2행 (이 절만을 위한 간단 버전 — 컬럼 3개뿐)
# 📖 문법 참고: INSERT INTO ... VALUES (...), (...), (...)처럼 값 묶음을 쉼표로 이어 쓰면
#    한 번의 INSERT로 여러 행을 동시에 넣을 수 있습니다. NUMERIC(5,4)는 "전체 5자리 중
#    소수점 아래 4자리까지 정확히 저장하는 숫자" 타입입니다(예: 0.8700) — 부동소수점(float)과
#    달리 소수 계산에서 오차가 생기지 않습니다.
cur.execute("""
    INSERT INTO hourly_error_stats (log_date, hour, error_count) VALUES
      ('2016-11-09', 0, 7148),
      ('2016-11-09', 3, 61442),
      ('2016-11-09', 14, 12088),
      ('2016-11-09', 23, 4231)
""")
cur.execute("""
    CREATE TABLE spike_windows (
        id          SERIAL PRIMARY KEY,
        start_hour  SMALLINT NOT NULL,
        spike_ratio NUMERIC(5,4)
    )
""")
cur.execute("""
    INSERT INTO spike_windows (start_hour, spike_ratio) VALUES (3, 0.87), (14, 0.52)
""")
conn.commit()
print("hourly_error_stats 4행 + spike_windows(교육용) 2행 준비 완료")


hourly_error_stats 4행 + spike_windows(교육용) 2행 준비 완료


In [11]:
# ✅ INNER JOIN — 양쪽 모두에 있는 행만 (hour 0, 23은 spike_windows에 없으므로 제외)
# 📖 문법 참고: JOIN은 두 표를 어떤 조건(ON)으로 옆으로 이어붙이는 것입니다.
#    INNER JOIN은 "양쪽 표에 조건이 맞는 행이 둘 다 있을 때만" 결과에 남기고,
#    LEFT JOIN은 "왼쪽 표(FROM 뒤)의 모든 행을 무조건 남기고, 오른쪽에 짝이 없으면 NULL로 채움"이
#    차이입니다. s.hour처럼 컬럼명 앞에 표 별명(s, w)을 붙이는 건 두 표에 같은 이름의 컬럼(hour)이
#    있을 때 어느 표의 것인지 구분하기 위해서입니다.
print("[INNER JOIN]")
run("""
    SELECT s.hour, s.error_count, w.spike_ratio
    FROM   hourly_error_stats s
    INNER JOIN spike_windows w ON s.hour = w.start_hour
    ORDER BY s.hour
""")

print()
print("[LEFT JOIN]")
run("""
    SELECT s.hour, s.error_count, w.spike_ratio
    FROM   hourly_error_stats s
    LEFT JOIN spike_windows w ON s.hour = w.start_hour
    ORDER BY s.hour
""")


[INNER JOIN]
(3, 61442, Decimal('0.8700'))
(14, 12088, Decimal('0.5200'))
  → 2행

[LEFT JOIN]
(0, 7148, None)
(3, 61442, Decimal('0.8700'))
(14, 12088, Decimal('0.5200'))
(23, 4231, None)
  → 4행


[(0, 7148, None),
 (3, 61442, Decimal('0.8700')),
 (14, 12088, Decimal('0.5200')),
 (23, 4231, None)]

> 💡 출력에서 `spike_ratio`가 `Decimal('0.8700')`으로 보이는 이유: `NUMERIC` 타입은 부동소수점 오차 없이 정확한 값을 다루기 위해 psycopg가 파이썬 `decimal.Decimal`로 돌려줍니다(`float`이 아닙니다). 돈·비율처럼 정밀도가 중요한 값에 `NUMERIC`을 쓰는 이유이기도 합니다.

### ⚠️ 흔한 실수 — 쉼표로 이어붙이면 CROSS JOIN


In [ ]:
# ⚠️ FROM A, B는 조건이 없는 CROSS JOIN입니다 — 4행 × 2행 = 8행이 그대로 곱해집니다.
run("SELECT s.hour, w.start_hour FROM hourly_error_stats s, spike_windows w ORDER BY s.hour, w.start_hour") 


(0, 3)
(0, 14)
(3, 3)
(3, 14)
(14, 3)
(14, 14)
(23, 3)
(23, 14)
  → 8행


[(0, 3), (0, 14), (3, 3), (3, 14), (14, 3), (14, 14), (23, 3), (23, 14)]

### 📦 모듈 4-2 · B-tree 인덱스 만들기


In [13]:
# ✅ 포인트: hour 컬럼에 B-tree 인덱스를 생성합니다.
# 📖 문법 참고: CREATE INDEX 인덱스이름 ON 테이블 (컬럼)은 그 컬럼으로 조회할 때 테이블 전체를
#    다 훑지 않고 더 빠르게 찾을 수 있도록 별도의 "찾아보기 표"를 만드는 SQL입니다(책의 색인과 비슷).
#    pg_size_pretty(pg_relation_size(...))는 그 인덱스(또는 테이블)가 디스크에서 차지하는 용량을
#    사람이 읽기 좋은 단위(KB/MB 등)로 보여주는 PostgreSQL 함수입니다.
cur.execute("CREATE INDEX idx_hourly_stats_hour ON hourly_error_stats (hour)")
conn.commit()
run("SELECT pg_size_pretty(pg_relation_size('idx_hourly_stats_hour'))")


('16 kB',)
  → 1행


[('16 kB',)]

> ⚠️ 교안은 "100만 행 기준 약 21MB"를 예로 듭니다. 지금 `hourly_error_stats`는 4행뿐이라 인덱스도 PostgreSQL의 최소 크기인 **16 kB**로 나옵니다 — 숫자가 다른 게 정상입니다(4행짜리 인덱스가 21MB일 수는 없겠죠). 100만 행 규모의 실제 인덱스 크기는 6교시에서 `log_events`로 직접 확인합니다.

### 💡 핵심 요약

- `INNER JOIN` = 교집합, `LEFT JOIN` = 왼쪽 전체 보존(오른쪽 없으면 `NULL`)
- `FROM A, B`(쉼표)는 조건 없는 **CROSS JOIN** — 4×2=8행처럼 예상 밖으로 불어남
- 인덱스 크기는 행 수에 비례 — 지금의 16 kB는 4행 기준값, 100만 행 규모는 6교시에서 확인


---
## 5교시 · 옵티마이저의 판단 — pg_stats로 근거를 직접 봅니다

옵티마이저가 Index Scan/Seq Scan을 고를 때 참고하는 통계를 `pg_stats`에서 조회합니다. `ANALYZE`를 먼저 실행해야 통계가 채워집니다.


In [14]:
# ✅ 포인트: ANALYZE로 통계를 갱신한 뒤 pg_stats에서 n_distinct·correlation을 확인합니다.
# 📖 문법 참고: ANALYZE 테이블명은 "이 테이블의 데이터 분포를 다시 조사해서 통계를 갱신하라"는
#    명령입니다 — 옵티마이저(쿼리 실행 계획을 짜는 부분)가 이 통계를 보고 인덱스를 쓸지 말지 판단합니다.
#    pg_stats는 PostgreSQL이 관리하는 시스템 뷰(테이블처럼 조회할 수 있는 통계 정보)입니다.
#    n_distinct는 "값이 대략 몇 종류나 있는지", correlation은 "데이터가 저장된 물리적 순서와
#    그 컬럼값의 순서가 얼마나 일치하는지(-1~1)"를 뜻합니다.
cur.execute("ANALYZE hourly_error_stats")
conn.commit()
run("""
    SELECT attname, n_distinct, correlation
    FROM   pg_stats
    WHERE  tablename = 'hourly_error_stats' AND attname = 'hour'
""")


('hour', -1.0, 1.0)
  → 1행


[('hour', -1.0, 1.0)]

> ⚠️ 교안의 예시값(`n_distinct=24, correlation≈0.998`)과 다릅니다 — 교안 예시는 7교시에서 실제로 24시간 분(24행)을 채운 **이후**의 값이고, 지금 이 표는 4교시에서 넣은 **4행**(hour = 0,3,14,23)뿐이기 때문입니다. `n_distinct = -1`은 "표본이 아니라 전체가 서로 다른 값"이라는 뜻이고(4행 모두 hour가 다름), `correlation = 1`은 "삽입 순서와 물리적 저장 순서가 완전히 일치"한다는 뜻입니다(0→3→14→23 순서로 넣었으니 당연합니다). 7교시에서 24행이 채워진 뒤 이 쿼리를 다시 실행해 보면 교안의 예시값에 가까워지는 것을 볼 수 있습니다.

### 💡 핵심 요약

- `pg_stats`는 옵티마이저가 실제로 참고하는 통계표 — `n_distinct`(고유값 수 추정), `correlation`(물리적 정렬도) 등을 담습니다.
- 통계는 **표본과 분포에 따라 달라집니다** — 지금처럼 행이 적으면 교안의 "실전 규모" 예시값과 다르게 나오는 것이 정상입니다.
- 선택도(`WHERE hour=3`처럼 좁은 조건일수록 인덱스 유리)를 실제로 판단하려면 6교시처럼 **충분히 큰 표**가 필요합니다.


---
## 6교시 · 직접 재봅니다 — EXPLAIN ANALYZE 실측

### 📦 준비 — log_events (100만 행)

5교시의 4행짜리 표로는 옵티마이저의 진짜 판단을 보기 어렵습니다. `lab/load/data/init_log_events.sql`과 같은 스키마·데이터로 100만 행 테이블을 만듭니다(이미 실행해 뒀다면 재사용 — 재실행 안전).


In [20]:
# ✅ 포인트: lab/load/data/init_log_events.sql과 동일한 스키마·데이터 생성 로직. 이미 실행해 뒀다면 건너뜁니다.
# 아래는 Windows(cmd.exe) 기준입니다 — Linux/macOS는 바로 아래 주석 처리된 줄을 대신 쓰세요.
!docker exec db-pg psql -U postgres -tc "SELECT 1 FROM pg_class WHERE relname='log_events'" | findstr /C:"1" >NUL && echo log_events 이미 존재 — 재사용 || docker exec -i db-pg psql -U postgres -d course_db < lab/load/data/init_log_events.sql

# ---- Linux/macOS ----
# !docker exec db-pg psql -U postgres -tc "SELECT 1 FROM pg_class WHERE relname='log_events'" | grep -q 1 && echo "log_events 이미 존재 — 재사용" || docker exec -i db-pg psql -U postgres -d course_db < ../lab/load/data/init_log_events.sql


DROP TABLE
CREATE TABLE
INSERT 0 1000000
CREATE INDEX
CREATE INDEX
CREATE INDEX
ANALYZE


> 위 셀은 노트북이 `lab_dayA/notebooks/`에 있다는 전제로 `../lab/load/data/init_log_events.sql`을 가리킵니다. 노트북을 다른 위치로 옮겼다면 경로를 맞게 고치거나, 교안 6교시 안내대로 `docker exec -i db-pg psql -U postgres -d course_db < lab/load/data/init_log_events.sql`을 터미널에서 직접 실행한 뒤 이 노트북을 이어가세요. 처음 생성하면 수 초~수십 초 걸립니다.

### 📦 모듈 6-1 · EXPLAIN vs EXPLAIN ANALYZE


In [ ]:
# ✅ EXPLAIN은 예측만, EXPLAIN ANALYZE는 실제 실행까지 합니다.
# 📖 문법 참고: EXPLAIN 뒤에 SQL을 붙이면 그 SQL을 "실제로 실행하지 않고" 옵티마이저가 세운
#    실행 계획(어떤 방식으로 데이터를 찾을지)만 보여줍니다 — Seq Scan(테이블 전체 훑기)인지
#    Index Scan(인덱스로 찾기)인지가 여기 나옵니다.
!docker exec db-pg psql -U postgres -d course_db -c "EXPLAIN SELECT * FROM log_events WHERE hour = 3;"


### 📦 모듈 6-2 · 인덱스가 안 타는 3가지 조건

**가설 1 — 낮은 선택도(약 4.2%)에서는 인덱스를 쓴다**


In [ ]:
# 📖 문법 참고: EXPLAIN ANALYZE는 위 EXPLAIN과 달리 SQL을 실제로 한 번 실행해보고, 예측(estimate)과
#    실측(actual) 시간·행 수를 함께 보여줍니다 — 그래서 실행 계획이 실제로 얼마나 정확했는지 알 수 있습니다.
!docker exec db-pg psql -U postgres -d course_db -c "EXPLAIN ANALYZE SELECT * FROM log_events WHERE hour = 3;"


→ 가설 1 **확인** ✅ : 인덱스를 씁니다. 다만 노드 이름이 교안의 `Index Scan`이 아니라 **`Bitmap Heap Scan`**입니다 — 5교시 "더 알아보기"에서 예고한 대로, 선택도가 아주 낮지도 높지도 않은 중간 구간(4.2%)에서는 PostgreSQL이 인덱스로 후보를 먼저 모은 뒤 heap을 읽는 이 절충 전략을 고르는 경우가 흔합니다. 어느 쪽이든 **인덱스를 실제로 사용했다**는 결론은 같습니다 — 정확히 어떤 노드를 고르는지는 하드웨어·설정·PostgreSQL 버전에 따라 달라질 수 있습니다.

**가설 2 — 컬럼에 함수를 적용하면 인덱스가 무력화된다**


In [ ]:
# 📖 문법 참고: EXTRACT(HOUR FROM created_at)은 timestamp 컬럼에서 "시(hour)" 부분만 뽑아내는
#    함수입니다. 다만 hour 컬럼에 인덱스가 있어도, WHERE에 EXTRACT(...)처럼 컬럼을 함수로 감싸면
#    그 인덱스를 그대로 쓸 수 없습니다(인덱스는 원래 컬럼값 기준으로 만들어졌기 때문) — 이래서
#    인덱스가 있어도 Seq Scan이 나옵니다.
!docker exec db-pg psql -U postgres -d course_db -c "EXPLAIN ANALYZE SELECT * FROM log_events WHERE EXTRACT(HOUR FROM created_at) = 3;"


→ 가설 2 **확인** ✅ : `created_at`에 인덱스가 있는데도 **Seq Scan**입니다(여러 CPU 코어가 나눠 읽는 `Parallel Seq Scan`으로 나타났습니다 — 최신 PostgreSQL의 병렬 쿼리 기능이며, 인덱스를 안 쓴다는 결론은 교안과 동일합니다). `EXTRACT(HOUR FROM created_at)`가 인덱스에 저장된 `created_at` 원본 값과 다른 형태라, 인덱스를 그대로 쓸 수 없기 때문입니다.

**해결법 A — 함수형 인덱스**


In [ ]:
# 📖 문법 참고: CREATE INDEX ... ON 테이블 (EXTRACT(HOUR FROM 컬럼))처럼 컬럼이 아니라 "함수의
#    결과"에 인덱스를 만들 수도 있습니다(함수형 인덱스) — 이러면 WHERE에서 같은 함수를 쓸 때도
#    인덱스를 다시 쓸 수 있게 됩니다.
!docker exec db-pg psql -U postgres -d course_db -c "CREATE INDEX idx_log_events_hour_extracted ON log_events (EXTRACT(HOUR FROM created_at));"
!docker exec db-pg psql -U postgres -d course_db -c "EXPLAIN ANALYZE SELECT * FROM log_events WHERE EXTRACT(HOUR FROM created_at) = 3;"


함수형 인덱스를 만들자 78.5ms → 20.8ms로 돌아왔습니다(Seq/Parallel Seq Scan → Bitmap Heap Scan). **해결법 B — 범위 조건으로 재작성**(컬럼은 그대로, 비교만 범위로)도 확인합니다.


In [ ]:
!docker exec db-pg psql -U postgres -d course_db -c "EXPLAIN ANALYZE SELECT * FROM log_events WHERE created_at >= '2016-11-09 03:00:00' AND created_at < '2016-11-09 04:00:00';"


기존 `idx_log_events_created_at` 인덱스를 그대로 활용해 새 인덱스 없이도 빨라졌습니다. **가설 3 — 선택도가 높으면 인덱스가 있어도 Seq Scan이 맞다**

> ⚠️ 교안은 `error_count > 0`(전체의 99.9%)을 예로 듭니다. 그런데 실제 `init_log_events.sql`의 데이터 분포를 그대로 재현하면 `error_count > 0`은 **약 1.5%**만 해당합니다(약 1.5% 확률로만 0이 아닌 값을 넣기 때문입니다) — 즉 이 조건은 실제로는 선택도가 **낮아서** 오히려 인덱스를 씁니다. 아래에서 실측으로 확인한 뒤, **실제로 선택도가 높은 조건**(`hour < 20`, 같은 `idx_log_events_hour` 인덱스가 있음에도 전체의 83%를 차지)으로 가설 3을 다시 확인합니다.


In [ ]:
# 📖 문법 참고: count(*) FILTER (WHERE 조건)은 "조건을 만족하는 행만 세라"는 뜻입니다 —
#    COUNT(*)만 쓰면 전체 행 수를 세는데, FILTER를 붙이면 그 조건에 맞는 것만 골라서 셉니다.
#    round(숫자, 자릿수)는 소수점을 원하는 자리까지 반올림합니다.
!docker exec db-pg psql -U postgres -d course_db -c "SELECT count(*) FILTER (WHERE error_count > 0) AS error_count_yangsu, count(*) AS total, round(100.0*count(*) FILTER (WHERE error_count > 0)/count(*), 2) AS pct FROM log_events;"
!docker exec db-pg psql -U postgres -d course_db -c "EXPLAIN ANALYZE SELECT * FROM log_events WHERE error_count > 0;"


In [ ]:
# ✅ 실제로 선택도가 높은 조건 — hour < 20 (전체의 약 83%). 같은 idx_log_events_hour 인덱스가 있어도 옵티마이저가 Seq Scan을 고릅니다.
!docker exec db-pg psql -U postgres -d course_db -c "EXPLAIN ANALYZE SELECT * FROM log_events WHERE hour < 20;"


→ 가설 3 **확인** ✅ : `hour`에 인덱스가 있는데도(가설 1에서 그 인덱스를 실제로 썼습니다) 조건을 `hour < 20`(83%)으로 바꾸자 옵티마이저가 **의도적으로 인덱스를 포기**하고 Seq Scan을 택했습니다 — 인덱스로 83만 번 흩어진 페이지를 오가는 것보다, 테이블을 처음부터 순서대로 한 번 읽는 게 더 빠르다는 판단입니다. **"Seq Scan = 느리다"는 틀린 공식**이라는 교안의 결론이 그대로 재현됩니다.

> 🔧 위에서 `error_count > 0`이 교안 설명과 다르게 나온 것은 **`lab/load/data/init_log_events.sql`의 실제 데이터 생성 로직이 교안 본문과 어긋나 있기 때문**입니다(교안은 error_count가 99.9%에서 양수라고 설명하지만, 실제 스크립트는 약 1.5%만 양수로 만듭니다). 강의 자료를 다듬을 때 참고하세요 — 이 노트북은 실제 데이터로 계속 진행하기 위해 `hour < 20`으로 가설 3을 대신 확인했습니다.

### 통계가 오래되면 예측이 틀어진다


In [ ]:
# ✅ INSERT 전 기준선
!docker exec db-pg psql -U postgres -d course_db -c "EXPLAIN ANALYZE SELECT * FROM log_events WHERE hour = 3;"


> ⚠️ 교안은 `lab/load/data/verify_analyze_scenario.sql`을 "03시에 40%가 쏠리는" 시나리오로 설명하지만, **실제 파일은 24시간에 균등하게(치우침 없이) 50만 행을 추가**합니다. 균등 추가는 교안이 스스로 경고한 대로("분포의 비율 자체가 바뀌어야 예측 오차가 뚜렷하게 드러납니다") 통계 오차를 거의 만들어내지 못합니다 — 아래에서 교안 6교시 본문에 실제로 실려 있는 **치우친 버전**(03시에 40% 쏠림)의 SQL을 그대로 실행합니다. `verify_analyze_scenario.sql` 파일도 이 버전에 맞춰 갱신이 필요해 보입니다.


In [ ]:
# ✅ 포인트: !로 시작하는 셸 명령은 Jupyter에서 한 줄이어야 합니다 — 교안 6교시 본문의 인라인 SQL을 한 줄로 옮겼습니다(내용은 동일, 03시에 40% 쏠리는 50만 행 추가).
# 📖 문법 참고: 이 문자열은 파이썬에서 여러 줄 문자열을 소괄호로 감싸 이어붙인 것뿐입니다(실제로는
#    한 줄짜리 SQL). SQL 안의 주요 문법: generate_series(1, 500000)은 1부터 50만까지 숫자를
#    하나씩 만들어내는 함수라 SELECT ... FROM generate_series(...)를 쓰면 그 개수만큼 행이
#    생성됩니다(50만 행짜리 INSERT를 만드는 트릭). CASE WHEN 조건 THEN 값1 ELSE 값2 END은
#    파이썬의 if/else와 같은 SQL 문법입니다. random()은 0~1 사이 난수를 돌려줍니다.
#    '문자열'::타입 은 캐스트(타입 변환) 문법입니다 — 예를 들어 '2016-11-10'::DATE는 그 문자열을
#    DATE 타입 값으로 바꿉니다. INTERVAL '59 minutes'는 "59분"이라는 시간 간격을 뜻하는 값이라
#    타임스탬프에 더하거나 뺄 수 있습니다.
sql = (
    "INSERT INTO log_events (log_date, created_at, hour, level, error_count, server_id, latency_ms) "
    "SELECT '2016-11-10'::DATE, "
    "'2016-11-10'::TIMESTAMP + (CASE WHEN random() < 0.4 THEN 3 ELSE (random()*23)::SMALLINT END || ' hours')::INTERVAL + (random() * INTERVAL '59 minutes'), "
    "CASE WHEN random() < 0.4 THEN 3 ELSE (random()*23)::SMALLINT END, "
    "CASE WHEN random() < 0.015 THEN 'ERROR' ELSE 'INFO' END, "
    "CASE WHEN random() < 0.015 THEN (random()*100)::INTEGER ELSE 0 END, "
    "(random()*10)::INTEGER + 1, "
    "(random()*200 + 10)::NUMERIC(10,2) "
    "FROM generate_series(1, 500000);"
)
cur.execute(sql)
conn.commit()
print("INSERT 0 500000  (50만 행 추가 완료)")


In [ ]:
# INSERT 후 / ANALYZE 전 — 예측이 아직 옛 통계를 기준으로 이뤄집니다
!docker exec db-pg psql -U postgres -d course_db -c "EXPLAIN ANALYZE SELECT * FROM log_events WHERE hour = 3;"


In [ ]:
# ANALYZE 실행 후 같은 쿼리 재실행 — 예측이 실제에 훨씬 가까워집니다
!docker exec db-pg psql -U postgres -d course_db -c "ANALYZE log_events;"
!docker exec db-pg psql -U postgres -d course_db -c "EXPLAIN ANALYZE SELECT * FROM log_events WHERE hour = 3;"


**ANALYZE 전후 비교** — 실측:

| 시점 | 예측 rows | 실제 rows | 예측 오차 |
| --- | --- | --- | --- |
| ANALYZE 전 | 61,350 | 254,709 | 약 4.2배 과소예측 |
| ANALYZE 후 | 254,800 | 254,709 | 0.04% |

실행 노드(`Bitmap Heap Scan`)는 바뀌지 않았지만(선택도 약 17%로 15~20% 경계 바로 위), `ANALYZE` 한 줄로 예측 정확도가 극적으로 좋아지는 것을 직접 확인했습니다 — 교안이 설명하는 "통계가 오래되면 예측이 틀어진다"는 원리 그대로입니다.

### 💡 핵심 요약

- **가설 1**: 낮은 선택도(4.2%) → 인덱스 사용(Bitmap Heap Scan) — 노드 이름은 환경에 따라 Index Scan일 수도, Bitmap Heap Scan일 수도 있습니다.
- **가설 2**: 컬럼에 함수 적용 → 인덱스 무력화(Seq/Parallel Seq Scan) → 함수형 인덱스 또는 범위 재작성으로 해결
- **가설 3**: 선택도가 높으면(여기서는 `hour < 20`, 83%) 인덱스가 있어도 Seq Scan이 더 빠른 선택
- 대량 INSERT 후 `ANALYZE`를 하지 않으면 예측이 크게 틀어질 수 있다 — `ANALYZE 테이블명;`으로 해결
- 🔧 이번 절에서 교안 본문·`init_log_events.sql`·`verify_analyze_scenario.sql` 사이에서 실제로 다른 점 두 가지(가설 3의 `error_count` 분포, 통계 드리프트 시나리오의 쏠림 여부)를 발견했습니다 — 위 ⚠️ 콜아웃에 각각 표시해 뒀습니다.


---
## 정리

이 노트북에서 2~6교시의 핵심 개념을 실제 PostgreSQL 위에서 확인했습니다:

| 교시 | 확인한 것 |
| --- | --- |
| 2 | 파일 기반 검색은 키 형식이 흐트러지면 **조용히 틀린 결과**를 낸다 |
| 3 | `CHECK` 제약이 실제로 잘못된 값을 거부한다 (+ psycopg 오류 후 `rollback()` 필요) |
| 4 | `INNER`/`LEFT JOIN` 차이, `FROM A,B`의 CROSS JOIN 위험, B-tree 인덱스 생성 |
| 5 | `pg_stats`가 옵티마이저의 판단 근거(n_distinct·correlation)를 담고 있다 |
| 6 | 인덱스가 안 타는 3가지 조건(함수 적용·높은 선택도·오래된 통계)을 실측으로 확인 |

다음은 **7교시**입니다 — 여기서 확인한 원리 위에서, 실제 로그 분석 결과를 `hourly_error_stats`·`latency_stats`·`spike_windows`에 안전하게 적재합니다(`lab_dayA/lab/load/` 실습, 또는 `PostgreSQL_3_로그파이프라인_개인실습.ipynb`).

### 🧹 정리(선택) — 이 노트북에서 만든 데모 테이블 지우기

7~8교시 실습을 이어서 할 계획이라면 **실행하지 마세요** — 7교시 스크립트가 어차피 `DROP TABLE IF EXISTS`로 정리하고 다시 만듭니다. 이 노트북만 독립적으로 테스트해 본 것이고 흔적을 지우고 싶다면 아래 셀을 실행하세요.


In [ ]:
# 🧹 선택 실행 — 이 노트북이 만든 데모 테이블을 정리합니다 (7교시로 이어갈 계획이면 실행하지 않아도 됩니다)
# cur.execute("DROP TABLE IF EXISTS spike_windows")
# cur.execute("DROP TABLE IF EXISTS hourly_error_stats")
# cur.execute("DROP TABLE IF EXISTS log_events")
# conn.commit()
# print("데모 테이블 정리 완료")
print("필요할 때 위 주석을 해제하고 실행하세요.")
